# EUI dataframe
---
This is a jupyter notebook to extract the information from the observations from the https://sidc.be/EUI/data/states/ and create a pandas dataframe that can be easily handled to look for observations with specific characteristics.

### Libraries

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from IPython.display import display, HTML

### Functions

In [2]:
def dataframe_from_eui():
    # URL
    url = "https://sidc.be/EUI/data/states/"
    short_names = ["HRI174", "FSI174", "FSI304"]

    # Fetch the webpage content
    response = requests.get(url)
    
    # Parse the HTML content with BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')
    
    # Extract all the rows
    rows = soup.find_all('tr')[1:]  
    
    # Splitting the rows in different columns
    data = []            
    for row in rows:
        row_data = []
        
        for cell in row.find_all('td'):
            links = cell.find_all('a', class_='broadcaster')
            if links:
                # Extract all href links and display them with a shortcut
                link_info = []
                if len(links) == 1:    # EUI-HRI
                    kk = 0
                else:                  
                    kk = 1             # FSI 174 and FSI 304
                for link in links:
                    href = link['href']
                    shortcut = f'<a href="{url}{href}">{short_names[kk]}</a>'
                    link_info.append(shortcut)
                    kk += 1
                
                # Join multiple links with a separator 
                row_data.append("|".join(link_info))

            else:
                # Otherwise, just get the text from the cell
                row_data.append(cell.get_text(strip=True))
        
        
        # Skip table for HRI Lya
        if row_data: 
            if 'hrilya1216' in row_data[0]:
                continue
            else:
                # Append processed row data
                data.append(row_data)

    # Define headers 
    headers = ["dataname", "num files", "sequence", "context", "start", "end", 
               "cadence (s)", "distance (AU)", "SolO-EarthAngle", "duration (s)", 
               "crval1", "crval2", "soopname", "xposure", "filter", "target", 
               "nbin1", "gaincomb", "recstate", "datamin", "datamax", 
               "hglt_obs", "hgln_obs", "crlt_obs", "crln_obs", "comments"]

    # Create the DataFrame
    df = pd.DataFrame(data, columns=headers)

    # Conversion from string to numbers
    numeric_cols = ['num files', 'cadence (s)', 'distance (AU)', 'SolO-EarthAngle', 'duration (s)', 
                    'crval1', 'crval2', 'xposure', 'nbin1', 'datamin', 'datamax', 
                    'hglt_obs', 'hgln_obs', 'crlt_obs', 'crln_obs']
    df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric, errors='coerce')

    # Conversion from string to datetime
    df['start'] = pd.to_datetime(df['start'], errors='coerce')
    df['end'] = pd.to_datetime(df['end'], errors='coerce')
    
    return df

## Dataframe
---
We load the entire dataframe

In [3]:
df = dataframe_from_eui()

In [4]:
print(df.keys())

Index(['dataname', 'num files', 'sequence', 'context', 'start', 'end',
       'cadence (s)', 'distance (AU)', 'SolO-EarthAngle', 'duration (s)',
       'crval1', 'crval2', 'soopname', 'xposure', 'filter', 'target', 'nbin1',
       'gaincomb', 'recstate', 'datamin', 'datamax', 'hglt_obs', 'hgln_obs',
       'crlt_obs', 'crln_obs', 'comments'],
      dtype='object')


### Example 1

In [6]:
# We can now stablish some conditions to obtain a sub dataset. 
# For instance, here I impose angle between 25 and 35 and observations with more than 100 fits.
# You can also filter by distance (to get the highest resolution ones). Check the key of the dataframe.
test = df[(df['SolO-EarthAngle'] > 25) & (df['SolO-EarthAngle'] < 35)  &(df['num files'] > 100)]
print("There are", test.shape[0], "observations fullfiling your criteria.")
# -------------------------------------------------------------------------------------------
# We plot the dataframe as an HTML to be able to download the states for JHelioviewer
# Just click in the link of HRI174 and you will download the state for JHelioviewer and you
# can check the observations there.
html_output = test.to_html(escape=False)
display(HTML(html_output))

There are 42 observations fullfiling your criteria.


,dataname,num files,sequence,context,start,end,cadence (s),distance (AU),SolO-EarthAngle,duration (s),crval1,crval2,soopname,xposure,filter,target,nbin1,gaincomb,recstate,datamin,datamax,hglt_obs,hgln_obs,crlt_obs,crln_obs,comments
62,hrieuvopn,544,HRI174,FSI174|FSI304,2024-10-19 19:21:08.630,2024-10-19 19:30:11.297,1,0.4872,25.23,544,1485.398580,-640.412443,R_SMALL_HRES_HCAD_RS-burst,0.659,open,,1,combined,on,0.000000,138121.738600,7.801101,-22.678514,7.801101,298.652289,
66,hrieuvopn,545,HRI174,FSI174|FSI304,2024-10-19 19:11:08.000,2024-10-19 19:20:11.665,1,0.4871,25.23,545,1485.251060,-640.800858,R_SMALL_HRES_HCAD_RS-burst,0.659,open,,1,combined,on,0.000000,138121.738600,7.800504,-22.691089,7.800504,298.731399,
70,hrieuvopn,372,HRI174,FSI174|FSI304,2024-10-19 19:04:01.264,2024-10-19 19:10:12.035,1,0.4871,25.23,372,1483.769558,-640.923484,R_SMALL_HRES_HCAD_RS-burst,0.659,open,,1,combined,on,0.000000,138121.738600,7.800078,-22.700028,7.800078,298.787601,
71,hrieuvopn,720,HRI174,FSI174|FSI304,2024-10-19 00:46:00.268,2024-10-19 01:45:55.275,5,0.4768,25.23,3600,25.842167,-175.491269,L_SMALL_HRES_HCAD_Fast-Wind,3.650,open,,1,combined,on,0.000000,24937.594660,7.727377,-24.126205,7.727377,307.417887,
72,hrieuvopn,720,HRI174,FSI174|FSI304,2024-10-18 07:15:00.150,2024-10-18 08:14:55.173,5,0.4669,27.24,3600,718.255525,-102.702327,L_SMALL_HRES_HCAD_Fast-Wind,3.650,open,,1,combined,on,0.000000,24937.594660,7.643705,-25.581266,7.643705,315.589085,
73,hrieuvopn,720,HRI174,FSI174|FSI304,2024-10-17 22:30:00.217,2024-10-17 23:29:55.224,5,0.4619,29.43,3600,-163.009298,2144.497340,R_SMALL_HRES_MCAD_Polar-Observations,3.650,open,,1,combined,on,0.000000,24937.594660,7.596282,-26.342894,7.596282,319.636049,
74,hrieuvopn,720,HRI174,FSI174|FSI304,2024-10-17 11:25:00.273,2024-10-17 12:24:55.279,5,0.4556,29.43,3600,112.949595,-21.830882,L_SMALL_HRES_HCAD_Fast-Wind,3.650,open,,1,combined,on,0.000000,24937.594660,7.530424,-27.342465,7.530424,324.727423,
75,hrieuv174,320,HRI174,FSI174|FSI304,2024-10-16 15:09:02.258,2024-10-16 15:40:56.262,6,0.4441,31.82,1920,1163.663503,1584.330522,none,2.650,Aluminium_174_2,,1,combined,on,0.000000,34348.007410,7.391902,-29.276037,7.391902,333.931526,
76,hrieuv174,320,HRI174,FSI174|FSI304,2024-10-16 13:59:02.251,2024-10-16 14:30:56.357,6,0.4434,31.82,1920,1311.464161,-1251.721007,none,2.650,Aluminium_174_2,,1,combined,on,0.000000,34348.007410,7.383170,-29.391682,7.383170,334.457051,
77,hrieuv174,1012,HRI174,FSI174|FSI304,2024-10-15 18:00:12.242,2024-10-15 22:29:48.272,16,0.4321,34.44,16192,1814.710162,-386.283738,L_BOTH_HRES_HCAD_Major-Flare,2.000,Aluminium_174_2,,1,combined,on,0.000000,45511.111450,7.219739,-31.449282,7.219739,343.380371,
